In [ ]:
import numpy as np
import pandas as pd


In [ ]:
crime_df = pd.read_csv('../../범죄발생지_20260202223427.csv')
crime_df

In [ ]:
crime_df = crime_df.T
crime_df

In [ ]:
crime_df.columns = crime_df.iloc[0]

In [ ]:
crime_df.head()

In [ ]:
crime_df.info()
crime_df

In [ ]:
crime_df.head()

In [ ]:
crime_df = crime_df.iloc[1:,:]

In [ ]:
crime_df

In [ ]:
crime_df = crime_df.iloc[2:,:]
crime_df

In [ ]:
crime_df = crime_df.replace('-',0)
crime_df = crime_df.apply(pd.to_numeric, errors='ignore')
crime_df

In [ ]:
crime_df.columns[:1]

In [ ]:
cols = crime_df.columns.tolist()
cols[0] = '나라'
cols[1] = '행정구역'
cols[2] = '도시'
crime_df.columns = cols
crime_df

In [ ]:
crime_df = crime_df.drop(columns='나라')

In [ ]:
# 행정구역/도시 먼저 따로 보관
meta = crime_df[['행정구역', '도시']]

# 숫자 컬럼만 따로 떼서 숫자로 변환
nums = crime_df.drop(columns=['나라', '행정구역', '도시'], errors='ignore')
nums = nums.replace('-', 0)
nums = nums.apply(pd.to_numeric, errors='coerce').fillna(0)



# gpt...
# 같은 이름 컬럼(강력범죄, 폭력범죄 등) 합치기
nums = nums.T.groupby(level=0).sum().T


In [ ]:
meta

In [ ]:
nums

In [ ]:
crime_df = pd.concat([meta, nums], axis=1)

In [ ]:
crime_df = crime_df[['행정구역','도시','강력범죄','폭력범죄','지능범죄','풍속범죄']]

In [ ]:
crime_df['crime'] = crime_df[['강력범죄','폭력범죄','지능범죄','풍속범죄']].sum(axis=1)
crime_df

In [ ]:
crime_df['violence'] = crime_df[['강력범죄','폭력범죄']].sum(axis=1)
crime_df

In [ ]:
crime_df = crime_df.drop(columns=['강력범죄','폭력범죄','지능범죄','풍속범죄'],errors='ignore')

In [ ]:
crime_df

In [ ]:
crime_df = crime_df[['행정구역','도시','crime','violence']].rename(columns={
    '행정구역': 'sido',
    '도시': 'sigungu',
    # '강력범죄': 'crime',# crime <- 범죄 총계
    # '폭력범죄': 'violence' <- 강력 + 폭력
    # 강력범죄 -> violence로
    # crime -> 총계
})

In [ ]:
crime_df

In [ ]:

#population에는 서울특별시, 부산광역시 처럼 돼 있는데
#지금 이 crime.json에는 서울, 부산 이렇게 돼 있음.
# 그래서 이름을 통일 시키려고하는 코드
sido_long_map = {
    "서울": "서울특별시",
    "부산": "부산광역시",
    "대구": "대구광역시",
    "인천": "인천광역시",
    "광주": "광주광역시",
    "대전": "대전광역시",
    "울산": "울산광역시",
    "세종": "세종특별자치시",
    "경기": "경기도",       # 혹시 '경기'로 들어온 값 대비
    "경기도": "경기도",
    "강원": "강원특별자치도",
    "충북": "충청북도",
    "충남": "충청남도",
    "전북": "전북특별자치도",
    "전남": "전라남도",
    "경북": "경상북도",
    "경남": "경상남도",
    "제주": "제주특별자치도",
}

# map : Series에 있는 각 값(서울, 부산, 대구…)을 딕셔너리로 하나씩 찾아서 바꿔주는 함수
# 값이 "서울"이면 → 딕셔너리에서 "서울" 키를 찾아 "서울특별시"로 바꿈
# 값이 "부산"이면 → "부산광역시"로 바꿈

#.fillna(crime_df["sido"]) : map()을 쓰다가 생긴 NaN을 “원래 값으로 복구”하는 역할
crime_df["sido"] = crime_df["sido"].map(sido_long_map).fillna(crime_df["sido"])


In [ ]:
# 새로운 csv 파일로 저장. transform에서 json 형태로 바꾸기 위해서?
# 이건 좀 gpt 한테 도움 받았음 ㅎㅎ;;

import os
os.makedirs('data', exist_ok=True)
crime_df.to_csv('data/crime_clean.csv', index=False, encoding='utf-8-sig')

In [ ]:
# json 형태로 바꾸기
# 얘도 좀 gpt 한테 도움 받았음 ㅎㅎ;;
crime_df.to_json('data/crime.json', orient='records', force_ascii=False, indent=2)